# Mississippi 2024 Data Processing

Produces two output GeoJSON files:
1. `ms_blocks.geojson` — census block level
2. `ms_precincts.geojson` — precinct level

Both contain: `votes_dem`, `votes_rep`, `white_pop`, `black_pop`

**Sources:**
- Dem/Rep votes (blocks): L2 voter file (`voted_party_democratic`, `voted_party_republican`)
- White/Black population (blocks): 2020 Census PL 94-171 files (`ms000012020.pl` + `msgeo2020.pl`)
  — total population counts (all ages), not just registered voters
- Block geometries: 2020 Census TIGER/Line shapefiles (downloaded)
- Precinct votes: 2024 general election precinct shapefile (`G24PREDHAR`, `G24PRERTRU`)
- Precinct race: spatially aggregated from block-level Census PL data

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import requests, zipfile, io, os

BASE = 'Data/ms/'
DATA_DIR = 'Data/ms/'
os.makedirs(DATA_DIR, exist_ok=True)

## 0. Download Census TIGER 2020 Block Shapefile for Mississippi

MS state FIPS = 28. File cached to `Data/` so we only download once.

In [2]:
TIGER_ZIP  = DATA_DIR + 'blocks/tl_2020_28_tabblock20.zip'
TIGER_DIR  = DATA_DIR + 'blocks/'
TIGER_URL  = 'https://www2.census.gov/geo/tiger/TIGER2020/TABBLOCK20/tl_2020_28_tabblock20.zip'

if not os.path.exists(TIGER_DIR):
    print('Downloading TIGER block shapefile (~90 MB)...')
    r = requests.get(TIGER_URL, stream=True)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        zf.extractall(TIGER_DIR)
    print('Done.')
else:
    print('TIGER blocks already downloaded.')

Done.


In [3]:
blocks_geo = gpd.read_file(TIGER_DIR + 'tl_2020_28_tabblock20.shp')
blocks_geo['GEOID20'] = blocks_geo['GEOID20'].astype(str)
print('Block geometries:', blocks_geo.shape, '| CRS:', blocks_geo.crs)
print(blocks_geo[['GEOID20']].head())

Block geometries: (112241, 18) | CRS: EPSG:4269
           GEOID20
0  281210209012020
1  280619504022002
2  280890306002030
3  280159501001053
4  281210201013012


## 1. Load & clean L2 voter data (census block level)

Only keeping vote columns — race population comes from Census PL files (next section).

In [4]:
l2 = pd.read_csv(
    BASE + 'l2/MS_l2_2024_gen_stats_2020block.csv',
    dtype={'geoid20': str},
    usecols=['geoid20', 'voted_party_democratic', 'voted_party_republican'],
)

# Drop non-block rows (county-level fallback rows have geoid < 15 digits)
l2 = l2[l2['geoid20'].str.len() == 15].copy()
l2['geoid20'] = l2['geoid20'].str.zfill(15)

print('L2 rows after filtering:', len(l2))
print(l2[['geoid20', 'voted_party_democratic', 'voted_party_republican']].head())

L2 rows after filtering: 112241
           geoid20  voted_party_democratic  voted_party_republican
0  280010001011000                       0                       0
1  280010001011001                       0                       0
2  280010001011002                       2                       0
3  280010001011003                       7                       5
4  280010001011004                       0                       0


## 2. Parse Census PL 94-171 files for `white_pop` / `black_pop`

### 2a. Geography file — map LOGRECNO → GEOID20 at block level (SUMLEV 750)

In [5]:
GEO_FILE  = BASE + 'census_pl/msgeo2020.pl'
DATA_FILE = BASE + 'census_pl/ms000012020.pl'

# Pipe-delimited; no header row
# Col 2 = SUMLEV, Col 7 = LOGRECNO, Col 9 = 15-digit GEOID20 (block level)
geo_raw = pd.read_csv(GEO_FILE, sep='|', header=None, dtype=str)

geo_blocks = geo_raw[geo_raw[2] == '750'][[7, 9]].copy()
geo_blocks.columns = ['LOGRECNO', 'GEOID20']
geo_blocks['LOGRECNO'] = geo_blocks['LOGRECNO'].str.zfill(7)

print('Block geo rows:', len(geo_blocks))
print(geo_blocks.head())

Block geo rows: 112241
      LOGRECNO          GEOID20
56989  0056990  280010001011000
56990  0056991  280010001011001
56991  0056992  280010001011002
56992  0056993  280010001011003
56993  0056994  280010001011004


### 2b. Data file — extract White alone (col 7) and Black alone (col 8), join to GEOID20

In [6]:
# Pipe-delimited; no header row
# Col 4 = LOGRECNO, Col 7 = White alone (P0010003), Col 8 = Black alone (P0010004)
data_raw = pd.read_csv(DATA_FILE, sep='|', header=None, dtype=str)

pl_data = data_raw[[4, 7, 8]].copy()
pl_data.columns = ['LOGRECNO', 'white_pop', 'black_pop']
pl_data['LOGRECNO'] = pl_data['LOGRECNO'].str.zfill(7)
pl_data['white_pop'] = pd.to_numeric(pl_data['white_pop'], errors='coerce').fillna(0).astype(int)
pl_data['black_pop'] = pd.to_numeric(pl_data['black_pop'], errors='coerce').fillna(0).astype(int)

# Join geo (LOGRECNO → GEOID20) with race data
pl_race = geo_blocks.merge(pl_data, on='LOGRECNO', how='left')
pl_race = pl_race[['GEOID20', 'white_pop', 'black_pop']]

print('PL race rows:', len(pl_race))
print('Total white_pop:', pl_race['white_pop'].sum(), ' | Total black_pop:', pl_race['black_pop'].sum())
print(pl_race.head())

PL race rows: 112241
Total white_pop: 1658893  | Total black_pop: 1084481
           GEOID20  white_pop  black_pop
0  280010001011000          5          3
1  280010001011001          0          0
2  280010001011002          7          2
3  280010001011003         41         16
4  280010001011004          0          0


## 3. Block-level GeoJSON

Join TIGER geometries → Census PL race data → L2 vote data on `GEOID20`.

| Output column | Source |
|---|---|
| `votes_dem` | L2 `voted_party_democratic` — registered Dems who voted |
| `votes_rep` | L2 `voted_party_republican` — registered Reps who voted |
| `white_pop` | Census PL `P0010003` — White alone, total population |
| `black_pop` | Census PL `P0010004` — Black alone, total population |

In [7]:
l2_slim = l2.rename(columns={'geoid20': 'GEOID20'})

# Left join: TIGER blocks → Census PL race → L2 votes
merged_blocks = (
    blocks_geo[['GEOID20', 'geometry']]
    .merge(pl_race, on='GEOID20', how='left')
    .merge(l2_slim, on='GEOID20', how='left')
)

# Fill blocks with no match with 0
for col in ['white_pop', 'black_pop', 'voted_party_democratic', 'voted_party_republican']:
    merged_blocks[col] = merged_blocks[col].fillna(0).astype(int)

merged_blocks = merged_blocks.rename(columns={
    'voted_party_democratic': 'votes_dem',
    'voted_party_republican': 'votes_rep',
})

print('Merged blocks shape:', merged_blocks.shape)
print('Blocks with any votes:', (merged_blocks['votes_dem'] + merged_blocks['votes_rep'] > 0).sum(), '/', len(merged_blocks))
print(merged_blocks[['GEOID20', 'votes_dem', 'votes_rep', 'white_pop', 'black_pop']].head())

Merged blocks shape: (112241, 6)
Blocks with any votes: 67729 / 112241
           GEOID20  votes_dem  votes_rep  white_pop  black_pop
0  281210209012020          0          0          0          0
1  280619504022002          7          0          0         18
2  280890306002030          4          0          9         11
3  280159501001053          1          3         18          0
4  281210201013012          0          0          0          0


In [8]:
# Sanity checks
print('Total dem votes (blocks):', merged_blocks['votes_dem'].sum())
print('Total rep votes (blocks):', merged_blocks['votes_rep'].sum())
print('Total white_pop (blocks):', merged_blocks['white_pop'].sum(), ' (expect ~1.66M)')
print('Total black_pop (blocks):', merged_blocks['black_pop'].sum(), ' (expect ~1.08M)')
print('Null geometries:', merged_blocks.geometry.isna().sum())

Total dem votes (blocks): 350742
Total rep votes (blocks): 495816
Total white_pop (blocks): 1658893  (expect ~1.66M)
Total black_pop (blocks): 1084481  (expect ~1.08M)
Null geometries: 0


In [9]:
out_blocks = gpd.GeoDataFrame(merged_blocks, geometry='geometry', crs=blocks_geo.crs)
out_blocks.to_file('Data/ms/cleaned/ms_blocks.geojson', driver='GeoJSON')
print('Saved ms_blocks.geojson')

Saved ms_blocks.geojson


## 4. Precinct-level GeoJSON

- **Dem/Rep votes**: from the 2024 precinct shapefile (`G24PREDHAR`, `G24PRERTRU`).
- **White/Black population**: spatially aggregated from block-level Census PL data (block centroid → containing precinct).

In [10]:
prec_gdf = gpd.read_file(BASE + 'precincts/ms_2024_gen_all_prec.shp')
print('Precincts:', prec_gdf.shape, '| CRS:', prec_gdf.crs)
print(prec_gdf[['UNIQUE_ID','COUNTY','PRECINCT','G24PREDHAR','G24PRERTRU']].head())

Precincts: (1735, 32) | CRS: EPSG:32616
                                UNIQUE_ID COUNTY  \
0     ADAMS-:-DIST. 1, BELLEMONT PRECINCT  Adams   
1  ADAMS-:-DIST. 1, BY-PASS FIRE PRECINCT  Adams   
2    ADAMS-:-DIST. 1, COURTHOUSE PRECINCT  Adams   
3      ADAMS-:-DIST. 2, BEAU PRE PRECINCT  Adams   
4   ADAMS-:-DIST. 2, DUNCAN PARK PRECINCT  Adams   

                         PRECINCT  G24PREDHAR  G24PRERTRU  
0     Dist. 1, Bellemont Precinct         510         850  
1  Dist. 1, By-Pass Fire Precinct         415         199  
2    Dist. 1, Courthouse Precinct         185         358  
3      Dist. 2, Beau Pre Precinct         287         504  
4   Dist. 2, Duncan Park Precinct         344         397  


In [11]:
# Reproject to a common projected CRS for accurate spatial ops
# EPSG:32616 (UTM Zone 16N) — matches the precinct shapefile's native CRS
blocks_proj = out_blocks.to_crs('EPSG:32616')
prec_proj   = prec_gdf.to_crs('EPSG:32616')

print('Blocks CRS:', blocks_proj.crs)
print('Precincts CRS:', prec_proj.crs)

Blocks CRS: EPSG:32616
Precincts CRS: EPSG:32616


In [12]:
# Build block centroids with race population data
block_centroids = blocks_proj[['GEOID20', 'white_pop', 'black_pop']].copy()
block_centroids['geometry'] = blocks_proj.geometry.centroid
block_centroids = gpd.GeoDataFrame(block_centroids, geometry='geometry', crs='EPSG:32616')

print('Block centroids:', len(block_centroids))

Block centroids: 112241


In [13]:
# Spatial join: assign each block centroid to a precinct
joined = gpd.sjoin(block_centroids, prec_proj[['UNIQUE_ID', 'geometry']], how='left', predicate='within')

print('Blocks assigned to a precinct:', joined['UNIQUE_ID'].notna().sum())
print('Blocks with no precinct match:', joined['UNIQUE_ID'].isna().sum())

Blocks assigned to a precinct: 112040
Blocks with no precinct match: 201


In [14]:
# Assign unmatched blocks (boundary gaps) to nearest precinct
unmatched_mask = joined['UNIQUE_ID'].isna()
print(f'{unmatched_mask.sum()} unmatched blocks — assigning to nearest precinct...')

if unmatched_mask.sum() > 0:
    unmatched = joined[unmatched_mask].drop(columns=['index_right', 'UNIQUE_ID'])
    nearest = gpd.sjoin_nearest(
        unmatched,
        prec_proj[['UNIQUE_ID', 'geometry']],
        how='left'
    )
    joined.loc[unmatched_mask, 'UNIQUE_ID'] = nearest['UNIQUE_ID'].values

print('Still unmatched:', joined['UNIQUE_ID'].isna().sum())

201 unmatched blocks — assigning to nearest precinct...
Still unmatched: 0


In [15]:
# Aggregate race population to precinct level
prec_race = joined.groupby('UNIQUE_ID')[['white_pop', 'black_pop']].sum().reset_index()
print('Precinct race table:', prec_race.shape)
print(prec_race.head())

Precinct race table: (1735, 3)
                                UNIQUE_ID  white_pop  black_pop
0     ADAMS-:-DIST. 1, BELLEMONT PRECINCT       1863       1063
1  ADAMS-:-DIST. 1, BY-PASS FIRE PRECINCT        465       1153
2    ADAMS-:-DIST. 1, COURTHOUSE PRECINCT        870        231
3      ADAMS-:-DIST. 2, BEAU PRE PRECINCT        902        648
4   ADAMS-:-DIST. 2, DUNCAN PARK PRECINCT        918        982


In [16]:
# Select vote columns from precinct shapefile and merge
prec_slim = prec_proj[['UNIQUE_ID', 'COUNTYFP', 'COUNTY', 'PRECINCT', 'G24PREDHAR', 'G24PRERTRU', 'geometry']].copy()
prec_slim = prec_slim.rename(columns={
    'G24PREDHAR': 'votes_dem',
    'G24PRERTRU': 'votes_rep',
})

merged_prec = prec_slim.merge(prec_race, on='UNIQUE_ID', how='left')
merged_prec[['white_pop', 'black_pop']] = merged_prec[['white_pop', 'black_pop']].fillna(0).astype(int)

print('Precinct merge shape:', merged_prec.shape)
print(merged_prec[['UNIQUE_ID', 'votes_dem', 'votes_rep', 'white_pop', 'black_pop']].head())

Precinct merge shape: (1735, 9)
                                UNIQUE_ID  votes_dem  votes_rep  white_pop  \
0     ADAMS-:-DIST. 1, BELLEMONT PRECINCT        510        850       1863   
1  ADAMS-:-DIST. 1, BY-PASS FIRE PRECINCT        415        199        465   
2    ADAMS-:-DIST. 1, COURTHOUSE PRECINCT        185        358        870   
3      ADAMS-:-DIST. 2, BEAU PRE PRECINCT        287        504        902   
4   ADAMS-:-DIST. 2, DUNCAN PARK PRECINCT        344        397        918   

   black_pop  
0       1063  
1       1153  
2        231  
3        648  
4        982  


In [17]:
# Sanity checks
print('Total dem votes (precincts):', merged_prec['votes_dem'].sum())
print('Total rep votes (precincts):', merged_prec['votes_rep'].sum())
print('Total white_pop (precincts):', merged_prec['white_pop'].sum(), ' (expect ~1.66M)')
print('Total black_pop (precincts):', merged_prec['black_pop'].sum(), ' (expect ~1.08M)')

print('\nBlock-level totals for comparison:')
print('  Dem votes:', merged_blocks['votes_dem'].sum())
print('  Rep votes:', merged_blocks['votes_rep'].sum())
print('  White pop:', merged_blocks['white_pop'].sum())
print('  Black pop:', merged_blocks['black_pop'].sum())

Total dem votes (precincts): 466667
Total rep votes (precincts): 747744
Total white_pop (precincts): 1658893  (expect ~1.66M)
Total black_pop (precincts): 1084481  (expect ~1.08M)

Block-level totals for comparison:
  Dem votes: 350742
  Rep votes: 495816
  White pop: 1658893
  Black pop: 1084481


In [18]:
out_prec = gpd.GeoDataFrame(merged_prec, geometry='geometry', crs='EPSG:32616')
out_prec.to_file('Data/ms/cleaned/ms_precincts.geojson', driver='GeoJSON')
print('Saved ms_precincts.geojson')

Saved ms_precincts.geojson


In [19]:
print('Done!')
print(f'  ms_blocks.geojson    — {len(out_blocks)} census blocks')
print(f'  ms_precincts.geojson — {len(out_prec)} precincts')
print()
print('Block columns:   ', list(out_blocks.columns))
print('Precinct columns:', list(out_prec.columns))

Done!
  ms_blocks.geojson    — 112241 census blocks
  ms_precincts.geojson — 1735 precincts

Block columns:    ['GEOID20', 'geometry', 'white_pop', 'black_pop', 'votes_dem', 'votes_rep']
Precinct columns: ['UNIQUE_ID', 'COUNTYFP', 'COUNTY', 'PRECINCT', 'votes_dem', 'votes_rep', 'geometry', 'white_pop', 'black_pop']
